In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm

# 이미지 파일 경로
IMAGE_DIR = '../Data Folder/H&M dataset/images/' 

# ResNet50 모델 로드 (이미지 특징 추출기)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1]).to(device)
resnet.eval()

# 이미지 전처리 설정
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 상품 명단 가져오기
articles = pd.read_parquet('../Data Folder/H&M dataset/H&m parquet dataset/articles.parquet')
article_ids = articles['article_id'].unique()

image_embeddings = {}

# 루프 돌며 특징 추출
for aid in tqdm(article_ids[:5000]): 
    aid_str = str(aid).zfill(10)
    img_path = os.path.join(IMAGE_DIR, aid_str[:3], f"{aid_str}.jpg")
    
    if os.path.exists(img_path):
        try:
            img = Image.open(img_path).convert('RGB')
            img_t = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = resnet(img_t).squeeze().cpu().numpy()
            image_embeddings[int(aid)] = feat
        except:
            continue

np.save('image_embeddings.npy', image_embeddings) # 이 npy파일이 있어야 04_DL이 돌아갑니다.

100%|██████████| 5000/5000 [14:53<00:00,  5.60it/s]


✅ 이미지 특징 추출 완료! 이제 DL.ipynb를 다시 실행하세요.
